In [1]:
import pandas as pd
pd.set_option("display.max_columns", None)
import numpy as np
# Now you can import your function
#from sampling_split import sample_and_split
import sys
from pathlib import Path

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder



In [3]:
df = pd.read_parquet("../../data/model_ready/flights_model_ready.parquet")
df.head()

,FlightDate,Airline,Origin,Dest,CRSDepTime,CRSElapsedTime,Distance,Year,Quarter,Month,DayofMonth,DayOfWeek,Marketing_Airline_Network,Operated_or_Branded_Code_Share_Partners,DOT_ID_Marketing_Airline,IATA_Code_Marketing_Airline,Flight_Number_Marketing_Airline,Operating_Airline,DOT_ID_Operating_Airline,IATA_Code_Operating_Airline,Tail_Number,Flight_Number_Operating_Airline,OriginAirportID,OriginAirportSeqID,OriginCityMarketID,OriginCityName,OriginState,OriginStateFips,OriginStateName,OriginWac,DestAirportID,DestAirportSeqID,DestCityMarketID,DestCityName,DestState,DestStateFips,DestStateName,DestWac,DepTimeBlk,CRSArrTime,ArrTimeBlk,DistanceGroup,year,target,date,dep_hour,tmpf,vsby,sknt,p01i,relh,gust,DayOfWeek_num,month_sin,month_cos,dow_sin,dow_cos,Distance_std,tmpf_std,vsby_std,sknt_std,relh_std,gust_std,distance_origin_norm,has_precip,is_holiday,prev_day_failure_origin,is_morning_peak,is_evening_peak
0,2018-01-03,Endeavor Air Inc.,ATL,ABY,1037,60.0,145.0,2018,1,1,3,Wednesday,DL,DL_CODESHARE,19790,DL,3298,9E,20363,9E,N981EV,3298,10397,1039707,30397,"Atlanta, GA",GA,13,Georgia,34,10146,1014602,30146,"Albany, GA",GA,13,Georgia,34,1000-1059,1137,1100-1159,1,2018,Delayed,2018-01-03,10,27.0,10.0,6.000000,0.0,40.42,0.0,2,0.5,0.866025,0.974928,-0.222521,-1.181369,-1.795908,0.372,-0.268741,-1.032586,-0.532803,-506.21629,0,0,1,0,0
1,2018-01-04,Endeavor Air Inc.,ATL,ABY,1037,60.0,145.0,2018,1,1,4,Thursday,DL,DL_CODESHARE,19790,DL,3298,9E,20363,9E,N981EV,3298,10397,1039707,30397,"Atlanta, GA",GA,13,Georgia,34,10146,1014602,30146,"Albany, GA",GA,13,Georgia,34,1000-1059,1137,1100-1159,1,2018,On time,2018-01-04,10,24.1,10.0,14.166667,0.0,54.36,21.0,3,0.5,0.866025,0.433884,-0.900969,-1.181369,-1.949265,0.372,1.558113,-0.405039,1.744748,-506.21629,0,0,1,0,0
2,2018-01-05,Endeavor Air Inc.,ATL,ABY,1037,60.0,145.0,2018,1,1,5,Friday,DL,DL_CODESHARE,19790,DL,3298,9E,20363,9E,N8877A,3298,10397,1039707,30397,"Atlanta, GA",GA,13,Georgia,34,10146,1014602,30146,"Albany, GA",GA,13,Georgia,34,1000-1059,1137,1100-1159,1,2018,On time,2018-01-05,10,17.1,10.0,12.153846,0.0,55.51,0.0,4,0.5,0.866025,-0.433884,-0.900969,-1.181369,-2.319438,0.372,1.107853,-0.353268,-0.532803,-506.21629,0,0,1,0,0
3,2018-01-06,Endeavor Air Inc.,ATL,ABY,1037,59.0,145.0,2018,1,1,6,Saturday,DL,DL_CODESHARE,19790,DL,3298,9E,20363,9E,N8970D,3298,10397,1039707,30397,"Atlanta, GA",GA,13,Georgia,34,10146,1014602,30146,"Albany, GA",GA,13,Georgia,34,1000-1059,1136,1100-1159,1,2018,On time,2018-01-06,10,21.9,10.0,6.846154,0.0,57.02,0.0,5,0.5,0.866025,-0.974928,-0.222521,-1.181369,-2.065605,0.372,-0.079459,-0.285291,-0.532803,-506.21629,0,0,1,0,0
4,2018-01-07,Endeavor Air Inc.,ATL,ABY,1037,60.0,145.0,2018,1,1,7,Sunday,DL,DL_CODESHARE,19790,DL,3298,9E,20363,9E,N8980A,3298,10397,1039707,30397,"Atlanta, GA",GA,13,Georgia,34,10146,1014602,30146,"Albany, GA",GA,13,Georgia,34,1000-1059,1137,1100-1159,1,2018,On time,2018-01-07,10,21.9,10.0,7.384615,0.0,54.75,0.0,6,0.5,0.866025,-0.781831,0.623490,-1.181369,-2.065605,0.372,0.040993,-0.387482,-0.532803,-506.21629,0,0,1,0,0


In [4]:
df.columns

Index(['FlightDate', 'Airline', 'Origin', 'Dest', 'CRSDepTime',
       'CRSElapsedTime', 'Distance', 'Year', 'Quarter', 'Month', 'DayofMonth',
       'DayOfWeek', 'Marketing_Airline_Network',
       'Operated_or_Branded_Code_Share_Partners', 'DOT_ID_Marketing_Airline',
       'IATA_Code_Marketing_Airline', 'Flight_Number_Marketing_Airline',
       'Operating_Airline', 'DOT_ID_Operating_Airline',
       'IATA_Code_Operating_Airline', 'Tail_Number',
       'Flight_Number_Operating_Airline', 'OriginAirportID',
       'OriginAirportSeqID', 'OriginCityMarketID', 'OriginCityName',
       'OriginState', 'OriginStateFips', 'OriginStateName', 'OriginWac',
       'DestAirportID', 'DestAirportSeqID', 'DestCityMarketID', 'DestCityName',
       'DestState', 'DestStateFips', 'DestStateName', 'DestWac', 'DepTimeBlk',
       'CRSArrTime', 'ArrTimeBlk', 'DistanceGroup', 'year', 'target', 'date',
       'dep_hour', 'tmpf', 'vsby', 'sknt', 'p01i', 'relh', 'gust',
       'DayOfWeek_num', 'month_sin', 'mon

In [5]:
df = df[['FlightDate', 'Airline', 'Origin', 'Dest',
        'Distance', 'Year',  'Month', 
       'DayOfWeek',
       'DepTimeBlk',
       'DistanceGroup', 
       'dep_hour', 'tmpf', 'vsby', 'sknt', 'p01i', 'relh', 'gust',
       #'DayOfWeek_num', 
       'month_sin', 'month_cos', 'dow_sin', 'dow_cos',
       'Distance_std', 
      # 'tmpf_std', 'vsby_std', 'sknt_std', 'relh_std', 'gust_std', 
       'distance_origin_norm', 'has_precip', 'is_holiday',
       'prev_day_failure_origin', 'is_morning_peak', 'is_evening_peak','target']]

In [6]:
# Get the root project directory (two levels up from the notebook)
project_root = Path.cwd().parents[1]  # notebooks/ → project root
scripts_path = project_root / "scripts"
sys.path.append(str(scripts_path))

# Now you can import your function
from sampling_split import sample_and_split

In [7]:
# X_train, X_test, y_train, y_test = sample_and_split(
#     df,
#     total_sample=500_000,
#     max_origin_frac=0.05,
#     max_region_frac=0.3,
#     test_size=0.2,
#     output_dir="../../data/model_ready/sampled_splits",
#     random_state=42
# )

In [8]:
# train_df = pd.read_parquet("../../data/model_ready/sampled_splits/train.parquet")
# test_df = pd.read_parquet("../../data/model_ready/sampled_splits/test.parquet")
# print(f"train shape: {train_df.shape}")
# print(f"test shape: {test_df.shape}")

# X_train = train_df.drop(columns = "target", axis =1)
# y_train = train_df.target


# X_test = test_df.drop(columns = "target", axis =1)
# y_test = test_df.target

### Let's start to build up our model one feature at a time

In [9]:
# Encode target to integers
le_target = LabelEncoder()
df['target_enc'] = le_target.fit_transform(df['target'])


In [10]:
# Assume these are your columns
numeric_features = df.select_dtypes(include=np.number).columns.tolist()
numeric_features = [col for col in numeric_features if col != 'target_enc']

categorical_features = df.select_dtypes(include='object').columns.tolist()
categorical_features = [col for col in categorical_features if col != 'target']


In [11]:
from sklearn.feature_selection import f_classif

X_num = df[numeric_features]
y = df['target_enc']
X_num = X_num.fillna(0)



In [12]:
f_vals, p_vals = f_classif(X_num, y)
numeric_corr = pd.Series(f_vals, index=numeric_features).sort_values(ascending=False)

In [13]:
from scipy.stats import chi2_contingency

def cramers_v(x, y):
    contingency = pd.crosstab(x, y)
    chi2 = chi2_contingency(contingency)[0]
    n = contingency.sum().sum()
    phi2 = chi2/n
    r,k = contingency.shape
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)
    return np.sqrt(phi2corr / min((kcorr-1), (rcorr-1)))

categorical_corr = {}
for col in categorical_features:
    categorical_corr[col] = cramers_v(df[col], df['target_enc'])

categorical_corr = pd.Series(categorical_corr).sort_values(ascending=False)


In [14]:
all_corr = pd.concat([numeric_corr, categorical_corr])
all_corr = all_corr.sort_values(ascending=False)
print("Top correlated features with target:")
print(all_corr.head(10))


Top correlated features with target:
dep_hour           181855.725445
is_morning_peak    116788.369786
is_evening_peak    113915.737200
has_precip          72789.695920
sknt                65737.244705
vsby                59050.770573
gust                53083.132313
tmpf                34742.363889
p01i                31415.140013
month_cos           17735.543132
dtype: float64


In [15]:
# Encode categorical features for RF
df_enc = pd.get_dummies(df[numeric_features + categorical_features], drop_first=True)
X = df_enc
y = df['target_enc']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)



In [ ]:
results = []

for i in range(1, 6):  # top 1 to 5 features
    top_features = all_corr.index[:i].tolist()
    
    # Handle categorical by including all one-hot columns
    selected_cols = []
    for f in top_features:
        if f in numeric_features:
            selected_cols.append(f)
        else:
            # all columns that start with f_
            selected_cols += [c for c in X.columns if c.startswith(f + "_")]
    
    X_train_sel = X_train[selected_cols]
    X_test_sel = X_test[selected_cols]
    
    #rf = RandomForestClassifier(n_estimators=80, random_state=42)
    rf = RandomForestClassifier(class_weight="balanced",
    n_estimators=80,
    max_depth=15,
    max_features="sqrt",
    n_jobs=-1)

    rf.fit(X_train_sel, y_train)
    y_pred = rf.predict(X_test_sel)
    
    report = classification_report(y_test, y_pred, output_dict=True)
    results.append({
        "num_features": i,
        "features": selected_cols,
        "accuracy": report['accuracy'],
        "f1_macro": report['macro avg']['f1-score'],
        "f1_weighted": report['weighted avg']['f1-score']
    })
    
    print(f"\n--- Top {i} features ---")
    print("Features:", selected_cols)
    print(classification_report(y_test, y_pred))



--- Top 1 features ---
Features: ['dep_hour']
              precision    recall  f1-score   support

           0       0.00      0.00      0.00     55020
           1       0.24      0.70      0.36    563410
           2       0.85      0.49      0.62   2289420

    accuracy                           0.52   2907850
   macro avg       0.37      0.39      0.33   2907850
weighted avg       0.72      0.52      0.56   2907850


--- Top 2 features ---
Features: ['dep_hour', 'is_morning_peak']
              precision    recall  f1-score   support

           0       0.00      0.00      0.00     55020
           1       0.24      0.70      0.36    563410
           2       0.85      0.49      0.62   2289420

    accuracy                           0.52   2907850
   macro avg       0.37      0.39      0.33   2907850
weighted avg       0.72      0.52      0.56   2907850


--- Top 3 features ---
Features: ['dep_hour', 'is_morning_peak', 'is_evening_peak']
              precision    recall  f1-sc

In [ ]:
results_df = pd.DataFrame(results)
print("\nSummary of RF performance with top N features:")
print(results_df)


In [ ]:
stop

### Let's tkeep class balances parameter, but play around with the features

In [ ]:

X_train = train_df.drop(columns="target")
y_train = train_df.target

X_test = test_df.drop(columns="target")
y_test = test_df.target

In [ ]:
FEATURES = ["Quarter", "Month", "DayofMonth", "DayOfWeek", "dep_hour", "DepTimeBlk",
    "Airline", "Origin", "Dest", "Distance", "DistanceGroup",
    "tmpf", "vsby", "sknt", "p01i", "relh", "gust"
]

TARGET = "target"

# Split categorical vs numerical
categorical_features = [
    "DepTimeBlk", "Airline", "Origin", "Dest", "DistanceGroup", "DayOfWeek"
]

numeric_features = [
    "Quarter", "Month", "DayofMonth", "dep_hour",
    "Distance", "tmpf", "vsby", "sknt", "p01i", "relh", "gust"
]

In [ ]:
# Get their positions (needed for SMOTENC)
cat_idx = [X_train.columns.get_loc(col) for col in categorical_features]

In [ ]:

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)



In [ ]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTENC
from sklearn.ensemble import RandomForestClassifier

# SMOTENC to handle categorical features
smote = SMOTENC(categorical_features=cat_idx, random_state=42)

# Class-weighted Random Forest
rf = RandomForestClassifier(class_weight="balanced", n_estimators=80, random_state=42,
                                max_depth=15,
    max_features="sqrt", n_jobs=-1)

# Full pipeline
pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('smote', smote),
    ('rf', rf)
])


In [ ]:
# Convert categorical columns to string (needed for OneHotEncoder)
for col in categorical_features:
    X_train[col] = X_train[col].astype(str)
    val_df[col]   = val_df[col].astype(str)
    test_df[col]  = test_df[col].astype(str)

# # Fill numeric NaNs with 0
for col in numeric_features:
    X_train[col] = X_train[col].fillna(0)
    val_df[col]   = val_df[col].fillna(0)
    test_df[col]  = test_df[col].fillna(0)

# Optional: fill categorical NaNs with "missing"
for col in categorical_features:
    X_train[col] = X_train[col].fillna("missing")
    val_df[col]   = val_df[col].fillna("missing")
    test_df[col]  = test_df[col].fillna("missing")

In [ ]:
pipeline.fit(X_train, y_train)


Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Quarter', 'Month',
                                                   'DayofMonth', 'dep_hour',
                                                   'Distance', 'tmpf', 'vsby',
                                                   'sknt', 'p01i', 'relh',
                                                   'gust']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['DepTimeBlk', 'Airline',
                                                   'Origin', 'Dest',
                                                   'DistanceGroup',
                                                   'DayOfWeek'])])),
                ('smote',
                 SMOTENC(categorical_features=[38, 1, 2, 3, 41, 11],
                         random_state=42)),
                ('rf',
                 RandomForestClassifier(class_weight='balanced', max_depth=15,
                                        n_estimators=80, n_jobs=-1,
                                        random_state=42))])

In [ ]:
X_test = X_test.fillna(0)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

y_pred = pipeline.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix (counts):")
print(confusion_matrix(y_test, y_pred))

# Optional: normalized confusion matrix
cm_norm = confusion_matrix(y_test, y_pred, normalize='true')
print("Confusion Matrix (normalized):")
print(np.round(cm_norm, 3))


TypeError: ufunc 'isnan' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''